In [12]:



from dotenv import load_dotenv
import os

# Document loading and parsing
from langchain_community.document_loaders.generic import GenericLoader
from langchain_community.document_loaders.parsers import LanguageParser

# Text splitting
from langchain_text_splitters import (
    Language,
    RecursiveCharacterTextSplitter
)

# Vector store
from langchain_community.vectorstores import FAISS

# OpenAI integrations
from langchain_openai import (
    OpenAIEmbeddings,
    ChatOpenAI
)

# Memory
from langchain_classic.memory import ConversationSummaryMemory

# Conversational Retrieval Chain
from langchain_classic.chains import ConversationalRetrievalChain

load_dotenv()


True

In [13]:
from git import Repo

In [14]:
import sys
print(sys.executable)

/Users/deepnath/Documents/Agentic AI class/Realtime-Source-code-Analyser/.venv/bin/python


In [15]:
repo_path='./test_repo'
repo= Repo.clone_from("https://github.com/entbappy/End-to-end-Medical-Chatbot-Generative-AI",to_path=repo_path)


GitCommandError: Cmd('git') failed due to: exit code(128)
  cmdline: git clone -v -- https://github.com/entbappy/End-to-end-Medical-Chatbot-Generative-AI ./test_repo
  stderr: 'fatal: destination path './test_repo' already exists and is not an empty directory.
'

In [16]:
loader= GenericLoader.from_filesystem(repo_path,
                                      glob="**/*",
                                      suffixes=[".py"],
                                      parser=LanguageParser(language=Language.PYTHON)
                                      )

In [17]:
documents= loader.load()

In [18]:
documents

[Document(metadata={'source': 'test_repo/setup.py', 'content_type': 'simplified_code', 'language': <Language.PYTHON: 'python'>}, page_content="from setuptools import find_packages, setup\n\nsetup(\n    name = 'Generative AI Project',\n    version= '0.0.0',\n    author= 'Bappy Ahmed',\n    author_email= 'entbappy73@gmail.com',\n    packages= find_packages(),\n    install_requires = []\n\n)"),
 Document(metadata={'source': 'test_repo/app.py', 'content_type': 'functions_classes', 'language': <Language.PYTHON: 'python'>}, page_content="def index():\n    return render_template('chat.html')"),
 Document(metadata={'source': 'test_repo/app.py', 'content_type': 'functions_classes', 'language': <Language.PYTHON: 'python'>}, page_content='def chat():\n    msg = request.form["msg"]\n    input = msg\n    print(input)\n    response = rag_chain.invoke({"input": msg})\n    print("Response : ", response["answer"])\n    return str(response["answer"])'),
 Document(metadata={'source': 'test_repo/app.py', 

In [19]:
character_text_splitter= RecursiveCharacterTextSplitter.from_language(
    chunk_size=500,
    chunk_overlap=20,
    language=Language.PYTHON)


chunks= character_text_splitter.split_documents(documents)



In [20]:
len(chunks)

17

In [21]:
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY")


from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

#embeddings= OpenAIEmbeddings(disallowed_special=())


/Users/deepnath/Documents/Agentic AI class/Realtime-Source-code-Analyser/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11652.42it/s]


In [22]:
from langchain_chroma import Chroma

vectordb = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory='./data'
)


In [23]:
from langchain_groq import ChatGroq

load_dotenv()

os.environ["GROQ_API_KEY"] = os.environ.get("GROQ_API_KEY")
llm = ChatGroq(
    model="qwen/qwen3.8-27b",
    temperature=0
)

In [24]:
retriever = vectordb.as_retriever(
    search_kwargs={"k": 4}
)

In [25]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

prompt = ChatPromptTemplate.from_template("""
You are a software engineering assistant.

Answer the question using only the provided context.

If the answer cannot be found in the context,
say that you don't have enough information.

Context:
{context}

Question:
{question}
""")

In [26]:
def format_docs(docs):
    return "\n\n".join(
        doc.page_content
        for doc in docs
    )

In [27]:
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
)

In [30]:
response = rag_chain.invoke(
    "what chat route does?"
)

print(response.content)

Based on the provided context, the `chat` function (associated with the `/get` route) performs the following actions:

1.  Retrieves the message from the request form using `request.form["msg"]`.
2.  Prints the input message.
3.  Invokes `rag_chain` with the message as the input.
4.  Prints the response answer.
5.  Returns the response answer as a string.
